In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
import os
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),  # Flip images randomly
    transforms.RandomRotation(15),
    transforms.ToTensor(),

])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),

])
train_dir = os.path.join(path,'PlantVillage', "train")
test_dir = os.path.join(path,'PlantVillage', "test")

train_dataset = ImageFolder(root=train_dir, transform=transform)

test_dataset  = ImageFolder(root=test_dir,  transform=transform)



batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels.shape}")


In [ ]:

classes=os.listdir(train_dir)
class_labels = {idx:label for idx,label in enumerate(classes)}
class_labels

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].set_title(class_labels[label])
    axes[i].axis('off')

plt.show()

In [ ]:
# Write your code here

import torch
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        """
        1️⃣ Define all layers in the model.
        """
        super(CustomModel, self).__init__()

        # Convolutional Layer + Activation + Pooling
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.Batch1 = nn.BatchNorm2d(16)


        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.Batch2 = nn.BatchNorm2d(32)



        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.Batch3 = nn.BatchNorm2d(64)


        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.Batch4 = nn.BatchNorm2d(128)


        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.Batch5 = nn.BatchNorm2d(256)



        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        # Fully Connected Layer
        self.fc = nn.Linear(256, 3)  # Output 10 classes

        # Softmax Layer
        self.softmax = nn.Softmax(dim=1)  # Apply along the class dimension

    def forward(self, x):
        """
        2️⃣ Define the forward pass (how data flows through the model).
        """
        x = self.conv1(x)  # Convolution
        x= self.Batch1(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = self.conv2(x)  # Convolution
        x= self.Batch2(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = self.conv3(x)  # Convolution
        x= self.Batch3(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = self.conv4(x)  # Convolution
        x= self.Batch4(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = self.conv5(x)  # Convolution
        x= self.Batch5(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling


        x = torch.flatten(x, start_dim=1)  # Flatten for FC layer
        x = self.fc(x)  # Fully connected layer
        x = self.softmax(x)  # Convert logits to probabilities
        return x  # Returns probability distribution



In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
# Write your code here

import torch.optim as optim



# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomModel()
model.to(device)


# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 2 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here

# Write your code here

import torch
import torch.nn as nn

class CustomModel2(nn.Module):
    def __init__(self):
        """
        1️⃣ Define all layers in the model.
        """
        super(CustomModel2, self).__init__()
        #[32,96,4,4]
        # Convolutional Layer + Activation + Pooling
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.Batch1 = nn.BatchNorm2d(16)


        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.Batch2 = nn.BatchNorm2d(32)



        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.Batch3 = nn.BatchNorm2d(64)


        self.conv4 = nn.Conv2d(in_channels=96, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.Batch4 = nn.BatchNorm2d(128)


        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.Batch5 = nn.BatchNorm2d(256)



        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


        self.pool_ = nn.MaxPool2d(kernel_size=8, stride=8)
        self.relu = nn.ReLU()
        # Fully Connected Layer
        self.fc = nn.Linear(256, 3)  # Output 10 classes

        # Softmax Layer
        self.softmax = nn.Softmax(dim=1)  # Apply along the class dimension

    def forward(self, x):
        """
        2️⃣ Define the forward pass (how data flows through the model).
        """
        x = self.conv1(x)  # Convolution
        x= self.Batch1(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling


        x = self.conv2(x)  # Convolution
        x= self.Batch2(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling
        res = self.pool(x)


        x = self.conv3(x)  # Convolution
        x= self.Batch3(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = torch.cat([x, res], dim=1)



        x = self.conv4(x)  # Convolution
        x= self.Batch4(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling


        x = self.conv5(x)  # Convolution
        x= self.Batch5(x)
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling


        x = torch.flatten(x, start_dim=1)  # Flatten for FC layer
        x = self.fc(x)  # Fully connected layer
        x = self.softmax(x)  # Convert logits to probabilities
        return x  # Returns probability distribution



In [ ]:
# Write your code here

import torch.optim as optim



# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomModel2()
model.to(device)


# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 2 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()